# ADV_data_tank_v01

Reads in Vectrino velocity data, filters it based on the Kraus and Strom despiking algorithms, then locates the decay region of the time series (with user input), and outputs a decay-only filtered time series to be used in calibration of the tank conditions.

Needs:
- Vectrino `*.dat` ASCII time series data

User input:
- Data folder
- Frequency at which the data was collected, e.g. 25 Hz
- The location in the time series where the mixer was turned off

## Imports

In [ ]:
# --- Find the project folder, then work inside its work/ folder ----------------
# All data paths below (data_OBS/, calibrations/, results/, ...) are relative to work/.
import os
import sys
_root = os.getcwd()
for _ in range(4):   # this folder, then up to three levels above
    if os.path.isdir(os.path.join(_root, 'tools')) and os.path.isdir(os.path.join(_root, 'work_template')):
        break
    _root = os.path.dirname(_root)
else:
    raise FileNotFoundError('could not find the project folder (the one containing tools/ and work_template/)')
sys.path.insert(0, _root)                          # for "from tools ..." imports
sys.path.insert(0, os.path.join(_root, 'tools'))   # for "import settling_tools"
if not os.path.isdir(os.path.join(_root, 'work')):
    from setup_work import setup_work
    setup_work(_root)                              # first run: create work/ from work_template/
os.chdir(os.path.join(_root, 'work'))
print('working in:', os.getcwd())
# ---------------------------------------------------------------------------------

# standard loads
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

# import the despiking functions from UKDE_despike_v02.py
from tools.UKDE_despike_v02 import *

%config InlineBackend.figure_format='retina' # high-res plots for a Retina display, uncomment while working

## Find the data files and prep for loading the files 

In [ ]:
##  --- User input ----------------------------- 

fs = 25 # frequency in Hz that the data was recorded at
dir_working = os.getcwd() # working directory, change to absolute path if running remote
dir_data = 'data_ADV/' # relative path to the data with respect to the working
dir_data_raw = 'data_ADV/raw/'

##   ------------------------------------------- 

ext = '*.dat'     # data file extension

fig_folder = dir_data+'figures'
# Make the output folder
if os.path.isdir(fig_folder) != 1:
    os.mkdir(fig_folder)

# Specify the column header names in the *.dat files. These names can be found in the .hrd file.

cnames = ['time_s','status','v1_m_s', 'v2_m_s','v3_m_s','v4_m_s',
          'amp1', 'amp2','amp3','amp4','snr1', 'snr2','snr3','snr4',
          'cor1', 'cor2','cor3','cor4']

# Find all of the data files and store the names 

files = sorted(glob.glob(dir_data_raw+ext))
nf=len(files)
os.chdir(dir_data_raw)
filenames = sorted(glob.glob(ext))
for i in range(0,nf):
    filenames[i]=os.path.splitext(os.path.basename(filenames[i]))[0] #strips extension
os.chdir(dir_working)
    
#Print the file names and index value for each file

for i in range(0,nf):
    print('file: ',filenames[i],'(index: ',i,')')

## Select the file to look at

In [ ]:
# --- User input ----------------------------- 
# select the file to look at by name; or set j to an index from the list above (e.g. j = 0)
j = filenames.index('ADV_02_60rpm_SAMPLE')
#  ------------------------------------------- 

print('file: ',filenames[j],'(index: ',j,')')

## Bring in the raw data and plot

In [ ]:
# read data from a single file into a pandas dataframe
data = pd.read_csv(files[j], sep=r'\s+', header=None) # reads in the data
data[0] = data[0].astype(float) #convert to float dtype to avoid warning
data.columns = cnames # renames the columns
data.iloc[:,0]=np.arange(len(data))/fs # changes the counter to time
data=data.drop(['status'], axis=1)  # delete the "status" column
print('number of rows =',len(data))

display(data.head(5)) # display the first five lines of the dataframe

# calculate the means
uavg = data.v1_m_s.mean()
vavg = data.v2_m_s.mean()
wavg = data.v3_m_s.mean()

print('mean u velocity [m/s] =',np.around(uavg,decimals=2))
print('mean v velocity [m/s] =',np.around(vavg,decimals=2))
print('mean w velocity [m/s] =',np.around(wavg,decimals=2))

# plot the velocity time series

fig, ax = plt.subplots(figsize=(6,3))
ax.plot(data.time_s,data.v1_m_s, alpha=0.75, label='u')
ax.plot(data.time_s,data.v3_m_s,alpha=0.75, label='w')
ax.plot(data.time_s,data.v2_m_s, alpha=0.75,label='v')
ax.set_xlabel('$t$ [s]')
ax.set_ylabel('$u,v,w$ [m/s]')
plt.legend(loc='best', ncol=1)
plt.savefig(fig_folder+'/'+filenames[j]+'_timeseries_unfiltered.pdf',bbox_inches="tight", pad_inches=0.005)

fig, ax = plt.subplots(figsize=(4,3))
plt.hist(data.v1_m_s, 50, density=True, alpha=0.5, label='u')
plt.hist(data.v2_m_s, 50, density=True, alpha=0.5, label='v')
plt.hist(data.v3_m_s, 50, density=True, alpha=0.5, label='w')
plt.xlabel('Velocity')
plt.ylabel('Probability')
plt.legend();
plt.savefig(fig_folder+'/'+filenames[j]+'_pdf_unfiltered.pdf',bbox_inches="tight", pad_inches=0.005)

## Filtering or despiking the data

In [ ]:
# Uses the UKDE despiking methods of Kraus 2024. 

##  --- User input ----------------------------- 

S_cut = 0.1 # gradient of the kernel where the cutoff is defined
R_cut = 0.1 # ratio of peak density to the density at S_cut

hu_METHOD = 'ROT_stdev' # options include:'IslamZhu','Chen_ures','ROT_stdev','ROT_Qn'
k_chen = 100 # coef if using the 'Chen_ures' hu_METHOD
ADV_FACE = 'side' # options are 'side' or 'down' for 4-beam Vectrino; 'singular and separate' when using despike_UKDE_1D

SHOW_PLOTS = 'yes' # to hide plots in notebook output use 'no'
SAVE_PLOTS = 'yes' # to not save the plots use 'no'
TITLE_PLOTS = 'yes' # to not title the plots use 'no'

plotpad = 0.05 # padding in output

time_cname = 'time_s'

##  ------------------------------------------- 

# define the output folders

velfilter_folder = dir_data+'/filtered'

# Make the output folders

if os.path.isdir(velfilter_folder) !=1:
    os.mkdir(velfilter_folder)

# setup the naming for the plots

file = j
datapt = filenames[j] 
plottitle = filenames[j]

# produce a filtered or despiked dataframe, i.e. dataf (which is the same as data but filtered)

dataf, Id_li, vels = despike_UKDE_xyz(data, fs, R_cut, S_cut, hu_METHOD, ADV_FACE, time_cname,
                 plotpad, plottitle, fig_folder, datapt, file,
                 SHOW_PLOTS, SAVE_PLOTS, TITLE_PLOTS)

# save filtered ascii

dataf.to_csv(velfilter_folder+'/'+filenames[j]+'_filtered.csv',index=False)


## Identify the decay portion of the time series

Identify the portion of the time series associated with the decay once the mixer has been turned off. The user will need to specify the `t1` value. `t1` is the time in seconds when the mixer was turned off within the ADV time series. The decay time series will then be saved for later processing.

In [ ]:
# user input -------------------------------------------------
t1 = 30 # time in seconds at which the mixer was turned off
# ------------------------------------------------------------

# extract the subsample and save the data 
dataf_decay = dataf[dataf['time_s'] > t1].copy()
dataf_decay.insert(dataf_decay.columns.get_loc('time_s') + 1,
                   'decay_time_s', dataf_decay['time_s'] - t1)

# calculate a surrogate for turbulent energy
dataf_decay['turb_energy'] = (dataf_decay['v1_m_s']**2+dataf_decay['v2_m_s']**2+dataf_decay['v3_m_s']**2)

# save the decay data to be used later in the tank calibration step
dataf_decay.to_csv(velfilter_folder+'/'+filenames[j]+'_filtered_decay_only.csv',index=False)

dataf_decay

## Visualize the decay

Plot of the turbulence surrogate.

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))
# ax.plot(dataf_decay.decay_time_s,dataf_decay['v1_m_s']**2, alpha=0.75, label='u')
# ax.plot(dataf_decay.decay_time_s,dataf_decay['v2_m_s']**2, alpha=0.75, label='v')
ax.plot(dataf_decay.decay_time_s,dataf_decay['v3_m_s']**2, alpha=0.75, label='w')
ax.plot(dataf_decay.decay_time_s,dataf_decay.turb_energy/3, alpha=0.75, label='avg')
ax.set_xlabel('$t$ [s]')
ax.set_ylabel('$u^2+v^2+w^2$ [m2/s2]')
ax.set_yscale('log')
plt.legend(loc='best', ncol=1);
# plt.savefig(fig_folder+'/'+filenames[j]+'_timeseries_unfiltered.pdf',bbox_inches="tight", pad_inches=0.005)


## Visualize the decay in turbulence relative to the background value

For building intuition about the decay time scale. The tank calibration will do its own calculation. 

The user should set `last_sec_for_avg` to the number of seconds at the end of the time series to average over for the background value. For example, if `last_sec_for_avg=60` the code will average the last 60 seconds of the time series to get the background turbulence value.

In [ ]:
# user input -------------------------------------------------
last_sec_for_avg = 60
# ------------------------------------------------------------

# calculate the background energy
background_energy = dataf_decay['turb_energy'].iloc[fs*last_sec_for_avg-1:].mean()

print('The average background turbuence level [m2/s2] is=',last_sec_for_avg)

fig, ax = plt.subplots(figsize=(6,3))
ax.plot(dataf_decay.decay_time_s,dataf_decay.turb_energy, alpha=0.75, label='avg')
ax.axhline(y=background_energy, color='C03', linestyle='--', linewidth=1, label='background')

ax.set_xlabel('$t$ [s]')
ax.set_ylabel('$u^2+v^2+w^2$ [m2/s2]')
# ax.set_yscale('log')
ax.set_ylim(0,0.005)
plt.legend(loc='best', ncol=1);